# 模块 4 视觉检测（`/detector`）—— 纯逻辑用法与效果展示

对应 `design/函数接口.md` 模块 4（`detect`）、`design/数据类型.md` 2.4（`DetectedTarget`/`Detection`）。

**职责**：订阅 `/camera/image_raw`，用 YOLOv8/v11 检测目标，输出 `/detections`。

本 notebook 只演示 **纯逻辑层**（`code/detector.py`），不涉及 ROS 节点壳；
节点壳的演示见 `detector_node.ipynb`。

> ### 与 `design/函数接口.md` 的一致性
> 文档定义的签名 `detect(image: np.ndarray) -> list[DetectedTarget]` **原样保留**；
> 模型路径等参数不在签名里，改由 `setup()` 注入（节点的 `__init__` 调用一次）。

> **依赖提示**：模块 4 需要 `ultralytics` + `torch`（CPU 版）。
> 项目基线是系统 numpy 1.17.4，而 ultralytics 要求 numpy>=1.18.5，
> 故视觉依赖装在独立目录 `~/m4_deps`，**不改动系统 numpy**（详见 `requirements.txt`）：
>
> ```bash
> pip3 install --target=~/m4_deps "numpy==1.21.6" torch==2.4.1 torchvision==0.19.1 ultralytics
> ```
>
> 本 notebook 中若未安装依赖，相关单元格会给出明确提示而不会中断整个演示。

## 0. 环境准备

In [ ]:
# 环境准备：把 code/ 加入 import 路径（notebook 位于 ai4/code/，直接同目录）
import os, sys
CODE_DIR = os.getcwd()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print("CODE_DIR =", CODE_DIR)

# 运行阶段开关（代码书写原则第 1 条）：dev 抛错 / onboard 兜底
# os.environ["UAV_STAGE"] = "onboard"
from config import STAGE, ONBOARD
print("STAGE =", STAGE, " ONBOARD =", ONBOARD)

## 1. 构造检测器

`YoloDetector` 持有模型资源，**模型惰性加载**（首次推理时才 import ultralytics 并载权重），
因此未安装依赖时本单元仍可执行。

In [ ]:
from detector import YoloDetector, DEFAULT_MODEL_PATH

det = YoloDetector(model_path=DEFAULT_MODEL_PATH, conf_thres=0.5, device="cpu")
print("model_path :", det.model_path)
print("conf_thres :", det.conf_thres)
print("iou_thres  :", det.iou_thres)
print("imgsz      :", det.imgsz)
print("device     :", det.device)
print("classes    :", det.classes, "（None/空 = 全部类别）")
print("max_det    :", det.max_det)

## 2. 单帧检测 `detect(image)`

输入约定：**BGR** 的 `np.ndarray`（与 ROS `sensor_msgs/Image` 的 `bgr8` 一致），
ultralytics 原生接受该布局，无需色彩空间转换。

输出约定：`bbox = (x, y, w, h)`（**左上角 + 宽高**），ultralytics 的 `xyxy` 已在内部转换。

In [ ]:
import numpy as np

# 若手头有真实图片，把路径填进来；否则用合成图（用于验证接口与空输入行为）
IMAGE_PATH = ""      # 例："/home/demo/data/target.jpg"

if IMAGE_PATH:
    import cv2
    image = cv2.imread(IMAGE_PATH)
    print("读入图片:", IMAGE_PATH, image.shape)
else:
    image = np.full((480, 640, 3), 127, dtype=np.uint8)
    print("使用合成图（纯灰）:", image.shape)

In [ ]:
try:
    detections = det.detect(image)
    print("检出目标数:", len(detections))
    for d in detections[:5]:
        print("  class_id=%d  confidence=%.3f  bbox=(x=%.1f, y=%.1f, w=%.1f, h=%.1f)"
              % (d.class_id, d.confidence, d.bbox[0], d.bbox[1], d.bbox[2], d.bbox[3]))
except RuntimeError as exc:
    print("[跳过] 未安装 ultralytics，无法真实推理：", exc)
    print("       请按上方『依赖提示』安装后重跑本单元。")

## 3. 结果可视化（方案书要求的交付物）

方案书原文：**"利用 AI 快速生成调用模型、处理图像矩阵、在画面上绘制 Bounding Box 并输出坐标数据"**。
绘制逻辑位于节点壳的 `draw_detections()`（保证纯逻辑层不引入 cv2），此处预览效果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches

# ---- 全项目统一配色（代码书写原则第 5 条）----
COLOR_BOX = "#00b050"      # bbox 框线
COLOR_CENTER = "#d62728"   # 画面中心参考线

try:
    from datatypes import DetectedTarget
    # 无模型（未安装依赖）时，退化为一条示例 bbox，使可视化仍然可见
    boxes = globals().get("detections") or [
        DetectedTarget(class_id=0, confidence=0.91, bbox=(280.0, 200.0, 120.0, 90.0))
    ]

    fig, ax = plt.subplots(figsize=(7, 5.25))
    ax.imshow(image[:, :, ::-1])                       # BGR -> RGB 供 matplotlib 显示
    for b in boxes:
        x, y, w, h = b.bbox
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False,
                                       edgecolor=COLOR_BOX, linewidth=2))
        ax.text(x, max(y - 6, 12), "cls%d %.2f" % (b.class_id, b.confidence),
                color=COLOR_BOX, fontsize=9)
    ax.axvline(image.shape[1] / 2, color=COLOR_CENTER, linestyle="--", linewidth=1)
    ax.axhline(image.shape[0] / 2, color=COLOR_CENTER, linestyle="--", linewidth=1)
    ax.set_title("detector: bbox (%d target(s))" % len(boxes))
    ax.set_xlabel("x [px]"); ax.set_ylabel("y [px]")
    plt.tight_layout(); plt.show()
except ImportError as exc:
    print("[跳过] 缺少 datatypes:", exc)

## 4. 空输入与边界（「空即正常」）

设计原则第 5 条：**空 = 没目标（业务正常）、静默 = 故障**。
`detect` 在无目标时返回 `[]`（不返回 `None`），节点壳据此发送**空** `Detections`。

In [ ]:
# 空数组 / 非法维数：onboard 阶段应告警并返回 []
import detector as det_mod

try:
    out = det_mod.detect(np.zeros((0, 0, 3), dtype=np.uint8))
    print("空数组 ->", out)
except RuntimeError as exc:
    print("[说明] 未调用 setup，模块级入口需先配置：", exc)

# 用实例直接验证边界
try:
    print("空数组        ->", det.detect(np.zeros((0, 0, 3), dtype=np.uint8)))
except ValueError as exc:
    print("空数组（dev 阶段抛错，符合预期）:", exc)

## 5. 阶段化参数校验（dev / onboard）

代码书写原则第 1 条：**开发阶段非法输入直接抛错**（问题尽早暴露），
**上机阶段告警并给默认值**（不因脏数据崩溃）。
由 `config.UAV_STAGE` 环境变量控制（默认 `dev`）。

In [ ]:
import importlib
import warnings

# --- dev：非法参数抛 ValueError ---
det_mod.ONBOARD = False
try:
    YoloDetector(model_path="")
    print("dev: 未抛错（异常）")
except ValueError as exc:
    print("dev   ->", exc)

# --- onboard：告警并回退默认值 ---
det_mod.ONBOARD = True
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    d2 = YoloDetector(model_path=None)
    print("onboard -> 告警:", [str(x.message) for x in w])
    print("onboard -> 回退后 model_path =", d2.model_path)

det_mod.ONBOARD = False      # 还原，避免影响其它单元

## 6. 小结

| 约定 | 取值 | 出处 |
|---|---|---|
| 输入图像 | BGR `np.ndarray` `(H, W, 3)` | ROS `bgr8` |
| 输出 bbox | `(x, y, w, h)` 左上角 + 宽高 | 数据类型.md 1.1 |
| 无目标 | 返回 `[]`，节点发空 `Detections` | 设计原则第 5 条 |
| 参数注入 | `setup(**kwargs)`，签名不含参数 | 函数接口.md 4.1 |
| 阶段校验 | `dev` 抛错 / `onboard` 兜底 | 代码书写原则第 1 条 |